In [104]:
#Rajout de 2021 dans la bdd car le fichier csv a été créé avec les bdd de 2022 à 2024

💾 Données 2021 ajoutées à paris_clean.csv


In [1]:
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

display(Markdown("# Nettoyage avancé des ventes d'appartements à Paris"))
df = pd.read_csv(r"C:\Users\User\Projet_Foncier\paris_clean.csv")

df = df[
    (df["Type local"].str.strip().str.lower() == "appartement") &
    (df["Nature mutation"] == "Vente") &
    (df["Valeur fonciere"] > 1000) &
    (df["Surface reelle bati"] > 9) &
    (df["Surface reelle bati"] < 1000)
].copy()

df["prix m2"] = df["Valeur fonciere"] / df["Surface reelle bati"]

q1 = max(df["prix m2"].quantile(0.05), 3000)
q3 = min(df["prix m2"].quantile(0.95), 20000)
df = df[(df["prix m2"] >= q1) & (df["prix m2"] <= q3)].copy()

z_scores = stats.zscore(df["prix m2"])
abs_z = abs(z_scores)
df = df[abs_z < 3]  

display(Markdown("## Vérification de la cohérence finale"))

pmin, pmax, pmoy, pstd = df["prix m2"].min(), df["prix m2"].max(), df["prix m2"].mean(), df["prix m2"].std()
print(f"Appartements conservés : {len(df):,}")
print(f"Prix min : {pmin:.0f} | max : {pmax:.0f} | moyenne : {pmoy:.0f} | écart-type : {pstd:.0f}")

suspicious = df[(df["prix m2"] < 3500) | (df["prix m2"] > 19000)]
print(f"\nBiens encore suspects : {len(suspicious)}")
if len(suspicious) > 0:
    display(suspicious[["Valeur fonciere", "Surface reelle bati", "prix m2", "Code postal", "Commune"]].head(5))
else:
    print("Aucune valeur aberrante détectée.")

output_path = r"C:\Users\User\Projet_Foncier\paris_appart_filtre.csv"
df.to_csv(output_path, index=False)


display(Markdown("## Aperçu des 50 premiers appartements filtrés"))
display(df[["Valeur fonciere", "Surface reelle bati", "prix m2", "Code postal", "Commune"]].head(50))


# Nettoyage avancé des ventes d'appartements à Paris

## Vérification de la cohérence finale

Appartements conservés : 81,362
Prix min : 5312 | max : 18121 | moyenne : 10531 | écart-type : 2445

Biens encore suspects : 0
Aucune valeur aberrante détectée.


## Aperçu des 50 premiers appartements filtrés

,Valeur fonciere,Surface reelle bati,prix m2,Code postal,Commune
1,605000.0,42.0,14404.761905,75003.0,PARIS 03
2,716250.0,69.0,10380.434783,75009.0,PARIS 09
3,320000.0,33.0,9696.969697,75010.0,PARIS 10
4,320000.0,29.0,11034.482759,75020.0,PARIS 20
5,220000.0,36.0,6111.111111,75020.0,PARIS 20
6,280000.0,28.0,10000.000000,75020.0,PARIS 20
7,200000.0,27.0,7407.407407,75019.0,PARIS 19
8,677500.0,58.0,11681.034483,75003.0,PARIS 03
9,445000.0,47.0,9468.085106,75019.0,PARIS 19
10,425890.0,52.0,8190.192308,75020.0,PARIS 20


On a gardé uniquement les ventes de maisons et appartements, avec surface bâtie, prix foncier et code postal. 
Les colonnes inutiles ou vides ont été supprimées.

In [2]:
import pandas as pd
from IPython.display import display, Markdown

display(Markdown("## Évolution du marché immobilier à Paris (Appartements uniquement, 2022–2024)"))

df_paris = pd.read_csv(r"C:\Users\User\Projet_Foncier\paris_clean.csv")

df_paris = df_paris.dropna(subset=["prix m2", "annee", "Valeur fonciere", "Surface reelle bati", "Type local"])

df_paris = df_paris[
    (df_paris["Type local"].str.strip().str.lower() == "appartement") &
    (df_paris["Surface reelle bati"] > 9) &
    (df_paris["Surface reelle bati"] < 1000) &
    (df_paris["prix m2"] > 1000) &
    (df_paris["prix m2"] < 30000) &
    (df_paris["Valeur fonciere"] > 1000) &
    (df_paris["Nature mutation"] == "Vente")
].copy()

display(Markdown("### Vérification de la cohérence des données"))
display(Markdown(f"#### Nombre total d'appartements conservés : {len(df_paris):,}"))

# On supprime les biens trop chers ou trop bas pour le marché parisien
df_paris = df_paris[(df_paris["prix m2"] >= 3000) & (df_paris["prix m2"] <= 20000)]

display(Markdown(f"#### \nBiens retirés pour valeurs extrêmes : {2588 - len(df_paris)} environ (estimation)"))

# Vérifier la plage des prix au m²
pmin, pmax, pmoy = df_paris["prix m2"].min(), df_paris["prix m2"].max(), df_paris["prix m2"].mean()
print(f"Prix au m² min : {pmin:.0f} | max : {pmax:.0f} | moyenne : {pmoy:.0f}")

# Détection de valeurs suspectes
suspicious = df_paris[(df_paris["prix m2"] < 2000) | (df_paris["prix m2"] > 20000)]
print(f"Biens suspects détectés : {len(suspicious)}")

if len(suspicious) > 0:
    print("\nExemples de biens suspects :")
    display(suspicious[["Valeur fonciere", "Surface reelle bati", "prix m2", "Code postal", "Commune"]].head(5))
else:
    print("Aucune valeur aberrante détectée.")

# ---Calcul des statistiques par année ---
stats_par_annee = {}
for annee in sorted(df_paris["annee"].unique()):
    subset = df_paris.loc[df_paris["annee"] == annee, "prix m2"]
    desc = subset.describe(percentiles=[0.25, 0.5, 0.75])
    stats_par_annee[annee] = {
        "Nombre de ventes": int(desc["count"]),
        "Prix moyen au m²": round(desc["mean"]),
        "Écart-type": round(desc["std"]),
        "1er quartile (25%)": round(desc["25%"]),
        "Médiane (50%)": round(desc["50%"]),
        "3e quartile (75%)": round(desc["75%"]),
    }

stats_valeurs = pd.DataFrame(stats_par_annee)
stats_valeurs.index.name = "Indicateur"

display(Markdown("###  Statistiques annuelles"))
display(stats_valeurs)

display(Markdown("###  Prix moyen au m² par arrondissement (appartements uniquement)"))

prix_arr = (
    df_paris.groupby("Code postal")["prix m2"]
    .mean()
    .round(0)
    .astype(int)
    .sort_index()
)

prix_arr_df = prix_arr.to_frame().T
prix_arr_df.index = ["Prix moyen au m²"]

display(prix_arr_df)


## Évolution du marché immobilier à Paris (Appartements uniquement, 2022–2024)

### Vérification de la cohérence des données

#### Nombre total d'appartements conservés : 87,405

#### 
Biens retirés pour valeurs extrêmes : -81801 environ (estimation)

Prix au m² min : 3000 | max : 20000 | moyenne : 10519
Biens suspects détectés : 0
Aucune valeur aberrante détectée.


###  Statistiques annuelles

,2022,2023,2024
Indicateur,,,
Nombre de ventes,33135,26836,24418
Prix moyen au m²,10968,10480,9953
Écart-type,2689,2768,2741
1er quartile (25%),9302,8706,8167
Médiane (50%),10759,10196,9625
3e quartile (75%),12400,11957,11385


###  Prix moyen au m² par arrondissement (appartements uniquement)

Code postal,75001.0,75002.0,75003.0,75004.0,75005.0,75006.0,75007.0,75008.0,75009.0,75010.0,75011.0,75012.0,75013.0,75014.0,75015.0,75016.0,75017.0,75018.0,75019.0,75020.0
Prix moyen au m²,12670,11703,12223,12908,12245,14159,13945,12468,11293,10099,10382,9642,9242,10115,10168,11544,10810,9618,8650,8918
